# Sentiment Feature Generation
Loads `Processed_Reviews.csv`, predicts sentiment for combined `Title + Text` using `cardiffnlp/twitter-roberta-base-sentiment`, appends only `Sentiment` as the final column, and writes a reproducibility report JSON (runtime, environment, warnings, row counts, and output SHA256).

In [ ]:
import os
import sys
import json
import time
import hashlib
import warnings
import platform
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import pandas as pd
import torch
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset

# Optional HF dataset iterator to improve GPU throughput in pipeline
runtime_installs = []
try:
    from datasets import Dataset
except ImportError:
    import subprocess

    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
    runtime_installs.append("datasets")
    from datasets import Dataset


def get_pkg_version(pkg_name):
    try:
        return version(pkg_name)
    except PackageNotFoundError:
        return None


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def build_output_path(input_file):
    input_abs = os.path.abspath(input_file)
    input_dir = os.path.dirname(input_abs)
    base_name = os.path.splitext(os.path.basename(input_abs))[0]
    out_name = f"{base_name}_with_Sentiment.csv"

    # Prefer writing beside the input file if possible.
    if os.path.isdir(input_dir) and os.access(input_dir, os.W_OK):
        return os.path.join(input_dir, out_name)

    # Then try Kaggle working directory, else current working directory.
    for candidate_dir in ["/kaggle/working", os.getcwd()]:
        try:
            os.makedirs(candidate_dir, exist_ok=True)
            if os.access(candidate_dir, os.W_OK):
                return os.path.join(candidate_dir, out_name)
        except Exception:
            pass

    return out_name


# Config
INPUT_FILE = "Processed_Reviews.csv"
BATCH_SIZE = 32
MAX_TEXT_LEN = 256
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"
MODEL_REVISION = None  # Set a fixed revision string for stricter reproducibility.
SAMPLE_SIZE = None  # Set e.g. 2000 for quick reproducibility checks.
PIPELINE_DEVICE = 0 if torch.cuda.is_available() else -1

OUTPUT_FILE = build_output_path(INPUT_FILE)
RUN_REPORT_FILE = OUTPUT_FILE.replace(".csv", "_run_report.json")

run_start = time.time()
run_start_utc = datetime.now(timezone.utc).isoformat()

# Load dataset
df = pd.read_csv(INPUT_FILE)
input_shape = [int(df.shape[0]), int(df.shape[1])]

if SAMPLE_SIZE is not None and SAMPLE_SIZE > 0:
    df = df.head(int(SAMPLE_SIZE)).copy()

# Build combined text from Title + Text
if "Title" not in df.columns and "Text" not in df.columns:
    raise ValueError("Input must contain at least one of: 'Title' or 'Text'.")

title = df["Title"].fillna("").astype(str) if "Title" in df.columns else pd.Series([""] * len(df))
text = df["Text"].fillna("").astype(str) if "Text" in df.columns else pd.Series([""] * len(df))
combined_text = (title + " " + text).str.strip()

# Prepare rows with actual text only
valid_indices = [i for i, t in enumerate(combined_text.tolist()) if isinstance(t, str) and t.strip()]
valid_texts = [combined_text.iloc[i] for i in valid_indices]

label_map = {"LABEL_0": "NEGATIVE", "LABEL_1": "NEUTRAL", "LABEL_2": "POSITIVE"}
predicted = [None] * len(combined_text)

captured_warnings = []
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    # Load sentiment model
    pipe_kwargs = {
        "task": "text-classification",
        "model": MODEL_NAME,
        "device": PIPELINE_DEVICE,
    }
    if MODEL_REVISION:
        pipe_kwargs["revision"] = MODEL_REVISION

    sentiment_pipe = pipeline(**pipe_kwargs)

    if valid_texts:
        # Use Dataset + KeyDataset for efficient batched inference.
        infer_ds = Dataset.from_dict({"text": valid_texts})
        outputs = sentiment_pipe(
            KeyDataset(infer_ds, "text"),
            truncation=True,
            max_length=MAX_TEXT_LEN,
            batch_size=BATCH_SIZE,
        )

        for idx, out in zip(valid_indices, outputs):
            raw_label = str(out.get("label", "")).strip().upper()
            predicted[idx] = label_map.get(raw_label, raw_label if raw_label else None)

    captured_warnings = [str(w.message) for w in caught]

# Append only this new column at the end
df["Sentiment"] = predicted

# Save
df.to_csv(OUTPUT_FILE, index=False)

run_end = time.time()
run_end_utc = datetime.now(timezone.utc).isoformat()
run_seconds = round(run_end - run_start, 3)

sentiment_counts = {str(k): int(v) for k, v in df["Sentiment"].value_counts(dropna=False).to_dict().items()}

run_report = {
    "run_start_utc": run_start_utc,
    "run_end_utc": run_end_utc,
    "run_seconds": run_seconds,
    "input_file": INPUT_FILE,
    "output_file": OUTPUT_FILE,
    "input_shape": input_shape,
    "output_shape": [int(df.shape[0]), int(df.shape[1])],
    "processed_rows": int(len(df)),
    "valid_text_rows": int(len(valid_texts)),
    "empty_text_rows": int(len(df) - len(valid_texts)),
    "config": {
        "batch_size": int(BATCH_SIZE),
        "max_text_len": int(MAX_TEXT_LEN),
        "model_name": MODEL_NAME,
        "model_revision": MODEL_REVISION,
        "sample_size": SAMPLE_SIZE,
        "pipeline_device": int(PIPELINE_DEVICE),
    },
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "gpu_available": bool(torch.cuda.is_available()),
        "gpu_count": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "package_versions": {
            "pandas": get_pkg_version("pandas"),
            "torch": get_pkg_version("torch"),
            "transformers": get_pkg_version("transformers"),
            "datasets": get_pkg_version("datasets"),
        },
    },
    "runtime_installs": runtime_installs,
    "warnings": captured_warnings,
    "sentiment_counts": sentiment_counts,
    "output_sha256": file_sha256(OUTPUT_FILE),
}

with open(RUN_REPORT_FILE, "w", encoding="utf-8") as f:
    json.dump(run_report, f, indent=2)

print("Saved output:", OUTPUT_FILE)
print("Saved run report:", RUN_REPORT_FILE)
print("Run time (seconds):", run_seconds)
print("Final shape:", df.shape)
print("Sentiment counts:", sentiment_counts)
print("Output sha256:", run_report["output_sha256"])
print(df[["Sentiment"]].head())